<a href="https://www.kaggle.com/code/neharaj1518/ai-mathematical-olympiad?scriptVersionId=357949654" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# Robust AIMO-style solver (crash-proof, safe integer parsing)
# - Reads test.csv with columns: id, problem
# - Writes submission.csv with columns: id, answer (integer 0..99999)
# - Per-problem process timeout, safe fallbacks, no int() crashes

import pandas as pd
import numpy as np
import re
import time
import math
import traceback
from concurrent.futures import ProcessPoolExecutor, TimeoutError, as_completed
from functools import partial

# sympy import (Kaggle usually has it)
try:
    import sympy as sp
except Exception:
    import sys
    !{sys.executable} -m pip install sympy
    import sympy as sp

# -------------------------
# Configuration
# -------------------------
TEST_CSV = "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv"
OUT_CSV = "submission.csv"
LOG_CSV = "solver_logs.csv"
TIME_LIMIT_SEC = 25.0      # seconds per problem
NUM_WORKERS = 4            # adjust for Kaggle resources

# -------------------------
# Safe helpers
# -------------------------
def try_int(val):
    """Safely try to convert to int. Returns None if fails."""
    try:
        return int(val)
    except Exception:
        try:
            # Sometimes val may be float-string
            return int(float(val))
        except Exception:
            return None

def clamp_int(x):
    """Clamp to integer within allowed [0,99999]."""
    try:
        xi = int(round(float(x)))
    except Exception:
        xi = 0
    if xi < 0:
        xi = 0
    if xi > 99999:
        xi = 99999
    return xi

def safe_int_from_text(s):
    """
    Extract strict digit-only integer tokens (1-5 digits) from text.
    Returns the most plausible integer or None.
    Avoids any token containing letters.
    """
    if s is None:
        return None
    # find pure digit tokens (1..5 digits)
    tokens = re.findall(r"\b\d{1,5}\b", str(s))
    if not tokens:
        return None
    # convert safely and choose a heuristic: largest token (common in problems)
    nums = [int(t) for t in tokens]
    return clamp_int(max(nums))

# -------------------------
# Pattern solvers (fast heuristics)
# -------------------------
def pattern_solvers(problem_text):
    p = str(problem_text).replace("\n", " ").strip()

    # explicit 'answer: 123' or 'Answer = 123'
    m = re.search(r"answer[s]?\s*[:=]\s*(\d{1,5})", p, re.I)
    if m:
        return clamp_int(int(m.group(1)))

    # explicit equals like '= 42' (common in placeholder problems)
    m = re.search(r"=\s*(\d{1,5})\b", p)
    if m:
        return clamp_int(int(m.group(1)))

    # factorial small '5!'
    m = re.search(r"(\d{1,3})\s*!", p)
    if m:
        n = int(m.group(1))
        if n <= 8:
            return clamp_int(math.factorial(n))

    # sum of first N integers like 'sum of the first 100 integers'
    m = re.search(r"sum of the first\s*(\d{1,6})\s*positive integers", p, re.I)
    if m:
        n = int(m.group(1))
        return clamp_int(n * (n + 1) // 2)

    # simple 1+2+...+n
    m = re.search(r"1\s*\+\s*2\s*\+\s*\.\.\.\s*\+\s*(\d{1,6})", p)
    if m:
        n = int(m.group(1))
        return clamp_int(n * (n + 1) // 2)

    # binomial \binom{n}{k}
    m = re.search(r"\\binom\{(\d{1,5})\}\{(\d{1,5})\}", p)
    if m:
        n, k = int(m.group(1)), int(m.group(2))
        try:
            return clamp_int(math.comb(n, k))
        except Exception:
            pass

    return None

# -------------------------
# Latex helpers & extraction
# -------------------------
def extract_binom_tokens(text):
    res = []
    for m in re.finditer(r"\\binom\{(\d{1,5})\}\{(\d{1,5})\}", str(text)):
        try:
            n, k = int(m.group(1)), int(m.group(2))
            res.append((n, k))
        except:
            continue
    return res

def extract_equation_sides(text):
    m = re.search(r"(.+?)=(.+)", str(text))
    if m:
        left = m.group(1).strip()
        right = m.group(2).strip()
        return left, right
    return None

# -------------------------
# Polynomial solver (sympy) - safe & limited
# -------------------------
def polynomial_integer_root_solver(problem_text):
    try:
        sides = extract_equation_sides(problem_text)
        if not sides:
            return None
        left, right = sides
        expr_text = f"({left})-({right})"
        expr_text = expr_text.replace("^", "**")
        # strip common LaTeX commands that break parsing (naive)
        expr_text = re.sub(r"\\[a-zA-Z]+\b", "", expr_text)
        expr_text = expr_text.replace("{", "(").replace("}", ")")
        # try common variable names
        for var in ['n','k','x','m','t']:
            try:
                v = sp.symbols(var)
                expr = sp.sympify(expr_text, evaluate=False)
                # only proceed if polynomial in v
                if expr.is_polynomial(v):
                    poly = sp.Poly(expr, v)
                    # try small integer roots first - limited range
                    const = int(poly.all_coeffs()[-1]) if poly.degree() >= 0 else 0
                    # small-range brute force (safe)
                    for r in range(-2000, 2001):
                        if poly.eval(r) == 0:
                            return clamp_int(r)
            except Exception:
                continue
    except Exception:
        pass
    return None

# -------------------------
# Combinatorics solver
# -------------------------
def combinatorics_solver(problem_text):
    p = str(problem_text)
    # simple "n choose k"
    m = re.search(r"(\d{1,4})\s+choose\s+(\d{1,4})", p, re.I)
    if m:
        n, k = int(m.group(1)), int(m.group(2))
        try:
            return clamp_int(math.comb(n, k))
        except Exception:
            pass
    bins = extract_binom_tokens(p)
    if bins:
        n, k = bins[0]
        try:
            return clamp_int(math.comb(n, k))
        except Exception:
            pass
    return None

# -------------------------
# Number theory - CRT heuristic
# -------------------------
def parse_mod_congruences(text):
    text = str(text)
    # find patterns like 'n ≡ a (mod m)' or 'n = a (mod m)'
    matches = re.findall(r"([nN])\s*[=≡]\s*(-?\d+)\s*\(?mod\s*(\d{1,5})\)?", text)
    eqs = []
    for _, a, m in matches:
        try:
            eqs.append((int(a), int(m)))
        except:
            continue
    return eqs

def crt_list(eqs):
    try:
        x, m_prod = 0, 1
        for a, m in eqs:
            a %= m
            g = math.gcd(m_prod, m)
            if (a - x) % g != 0:
                return None
            # find k such that x + k*m_prod ≡ a (mod m)
            found = False
            for k in range(0, m):
                if (x + k * m_prod) % m == a:
                    x = x + k * m_prod
                    m_prod = (m_prod // g) * m
                    x %= m_prod
                    found = True
                    break
            if not found:
                return None
        return x % m_prod
    except Exception:
        return None

def number_theory_solver(problem_text):
    eqs = parse_mod_congruences(problem_text)
    if eqs:
        sol = crt_list(eqs)
        if sol is not None:
            return clamp_int(sol)
    return None

# -------------------------
# Brute force small-range solver
# -------------------------
def brute_force_small(problem_text, time_budget=5.0):
    p = str(problem_text)
    # detect 'n between a and b' or '1 <= n <= 100'
    m = re.search(r"([a-zA-Z])\s+between\s+(\d{1,5})\s+and\s+(\d{1,5})", p, re.I)
    if m:
        var = m.group(1)
        lo, hi = int(m.group(2)), int(m.group(3))
    else:
        m2 = re.search(r"(\d{1,5})\s*≤\s*([a-zA-Z])\s*≤\s*(\d{1,5})", p)
        if m2:
            lo, hi = int(m2.group(1)), int(m2.group(3))
        else:
            # detect 'n <= 100' pattern
            m3 = re.search(r"([a-zA-Z])\s*(?:≤|<=)\s*(\d{1,5})", p)
            if m3:
                lo, hi = 1, int(m3.group(2))
            else:
                return None

    if hi - lo > 20000 or hi < lo:
        return None

    start = time.time()
    sides = extract_equation_sides(p)
    if sides:
        left, right = sides
        # find numeric right-hand side
        mnum = re.search(r"=\s*(-?\d+)\b", p)
        if mnum:
            target = int(mnum.group(1))
            for n in range(lo, hi + 1):
                if time.time() - start > time_budget:
                    break
                # attempt naive evaluation after replacing single-letter var with number
                expr = left.replace("^", "**")
                expr = re.sub(r"\\[a-zA-Z]+", "", expr)
                expr2 = re.sub(r"\b[a-zA-Z]\b", str(n), expr)
                try:
                    val = eval(expr2, {"__builtins__": {}}, {})
                    if isinstance(val, (int, float)) and int(round(val)) == target:
                        return clamp_int(n)
                except Exception:
                    continue
    return None

# -------------------------
# Main per-problem orchestration
# -------------------------
def solve_one(problem_tuple):
    """
    Input: (pid, problem_text)
    Output: (pid, answer_int, info_dict)
    """
    pid, text = problem_tuple
    info = {"id": pid, "picked_solver": None, "time": None, "error": None}
    start = time.time()

    try:
        # 1) pattern heuristics
        ans = pattern_solvers(text)
        if ans is not None:
            info["picked_solver"] = "pattern"
            info["time"] = time.time() - start
            return pid, clamp_int(ans), info

        # 2) combinatorics
        ans = combinatorics_solver(text)
        if ans is not None:
            info["picked_solver"] = "combinatorics"
            info["time"] = time.time() - start
            return pid, clamp_int(ans), info

        # 3) polynomial (sympy)
        ans = polynomial_integer_root_solver(text)
        if ans is not None:
            info["picked_solver"] = "polynomial_sympy"
            info["time"] = time.time() - start
            return pid, clamp_int(ans), info

        # 4) number theory (CRT)
        ans = number_theory_solver(text)
        if ans is not None:
            info["picked_solver"] = "number_theory"
            info["time"] = time.time() - start
            return pid, clamp_int(ans), info

        # 5) brute-force small-range
        ans = brute_force_small(text, time_budget=min(5.0, TIME_LIMIT_SEC * 0.5))
        if ans is not None:
            info["picked_solver"] = "brute_force_small"
            info["time"] = time.time() - start
            return pid, clamp_int(ans), info

        # 6) fallback literal extraction (strict)
        lit = safe_int_from_text(text)
        if lit is not None:
            info["picked_solver"] = "literal_extraction"
            info["time"] = time.time() - start
            return pid, clamp_int(lit), info

        # 7) final fallback
        info["picked_solver"] = "fallback_zero"
        info["time"] = time.time() - start
        return pid, 0, info

    except Exception as e:
        info["error"] = str(e) + "\n" + traceback.format_exc()
        info["picked_solver"] = "error_fallback"
        info["time"] = time.time() - start
        return pid, 0, info

# -------------------------
# Runner with per-task timeouts and safe logging
# -------------------------
def run_all_and_save():
    df = pd.read_csv(TEST_CSV)
    if "problem" not in df.columns:
        raise ValueError("test.csv must contain a 'problem' column with LaTeX text.")
    # ensure ids are strings but keep original order
    rows = [(r["id"], str(r["problem"])) for _, r in df.iterrows()]

    results = []
    logs = []

    # Submit all tasks to the pool and then collect with per-future timeout
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futures = []
        for row in rows:
            futures.append(ex.submit(solve_one, row))

        # iterate futures in submission order to map results back in input order
        for fut, row in zip(futures, rows):
            pid = row[0]
            try:
                res_pid, ans, info = fut.result(timeout=TIME_LIMIT_SEC)
            except TimeoutError:
                # worker timed out => fallback
                ans = 0
                info = {"id": pid, "picked_solver": "timeout", "time": TIME_LIMIT_SEC, "error": "Timeout"}
                # attempt to cancel (best-effort)
                try:
                    fut.cancel()
                except:
                    pass
            except Exception as e:
                ans = 0
                info = {"id": pid, "picked_solver": "exception", "time": None, "error": str(e)}
            # ensure integer and clamp
            ans = clamp_int(ans)
            results.append((pid, ans))
            logs.append(info)

    # Build output DataFrame aligned to original order
    res_df = pd.DataFrame(results, columns=["id", "answer"])
    # Align with original input order
    res_df = res_df.set_index("id").loc[[r[0] for r in rows]].reset_index()

    # Save submission (AIMO expects columns: id, answer)
    res_df.to_csv(OUT_CSV, index=False)

    # Save logs
    try:
        pd.DataFrame(logs).to_csv(LOG_CSV, index=False)
    except Exception:
        pass

    print(f"Wrote {OUT_CSV} and logs -> {LOG_CSV}")
    return res_df, pd.DataFrame(logs)

# -------------------------
# Entrypoint
# -------------------------
if __name__ == "__main__":
    t0 = time.time()
    print("Starting solver (robust). TIME_LIMIT_SEC =", TIME_LIMIT_SEC, "NUM_WORKERS =", NUM_WORKERS)
    submission_df, log_df = run_all_and_save()
    print("Done. Total time: %.2fs" % (time.time() - t0))
    print(submission_df.head())
    print(log_df.head())


Starting solver (robust). TIME_LIMIT_SEC = 25.0 NUM_WORKERS = 4
Wrote submission.csv and logs -> solver_logs.csv
Done. Total time: 0.11s
       id  answer
0  000aaa       1
1  111bbb       0
2  222ccc       4
       id       picked_solver      time error
0  000aaa  literal_extraction  0.005031  None
1  111bbb  literal_extraction  0.004997  None
2  222ccc             pattern  0.000995  None
